# Inteligência Artificial: Back-end
### Prova Prática

## Parte 1: Questão teórica

![texto alternativo](images\image.png)

O diagrama acima apresenta uma estrutura quase idêntica à estrutura base da DJI Cloud API que é utilizada para disponibilizar os serviços de drone da DJI por outras empresas. Entre as adaptações temos a implementação de um microsserviço abstraindo a comunicação com a Storage S3 para manter uma maior segurança nos dados, também foi inserido um Proxy Reverso (Gateway - Kong) para disponibilização de todos os serviços. E para fins de monitoramento, foram implementados serviços como o Prometheus e o Grafana. Com base no diagrama acima, descreva de maneira geral o objetivo de cada componente.

### Resposta

`Kong`: atua como Gateway e proxy reverso da arquitetura, centralizando a entrada das requisições, roteando chamadas para os serviços internos e aplicando autenticação, autorização, logs e controle de acesso. Também pode intermediar a comunicação via WebSocket.

`EMQX`: atua como broker MQTT, permitindo comunicação assíncrona entre serviços e dispositivos. É usado como sistema de mensageria para troca de eventos, comandos e dados de telemetria.

`Java API`: neste exemplo, representa a API central do sistema, pois se integra principalmente com o MySQL, usado para armazenar informações gerais como usuários, permissões e planos, e com o Redis, usado para sessões de usuário e dados frequentemente acessados que precisam de baixa latência.

`Redis`: utilizado como cache em memória, mantendo dados temporários e de acesso rápido, como sessões de login, bloqueios temporários de IP e informações frequentemente consultadas. Ajuda a reduzir consultas repetidas ao banco de dados.

`MySQL`: representa a camada de persistência dos dados estruturados do sistema, ou seja, o banco de dados principal da aplicação.

`API REST`: aparenta atuar como uma API intermediária, provavelmente utilizada como interface para acesso ao S3, permitindo buscar, enviar e gerenciar imagens, documentos e arquivos em geral.

`S3`: serviço da Amazon para armazenamento de objetos e arquivos em geral. É escalável, durável e adequado para armazenar imagens, vídeos, documentos e outros arquivos da plataforma.

`Prometheus`: funciona como uma base de métricas do sistema, coletando e armazenando informações relevantes das operações, como latência, número de requisições, erros e status dos componentes.

`Grafana`: funciona como a interface visual das métricas coletadas pelo Prometheus. Ele cria dashboards e visualizações que facilitam o monitoramento e a análise do comportamento do sistema.

`Usuário, controle remoto e drone`: aqui é a camada operacional o usuário ele interage com o controle que por sua vez interage com o sistema por meio do aplicativo android.

## Parte 2: Desenvolvimento de API RESTful

Crie uma API RESTful utilizando Flask ou Fast API que permita a criação, leitura, atualização e exclusão (CRUD) de registros de usuários. Cada usuário deve ter os seguintes campos:

- ID
- Nome
- E-mail
- Data de criação

Requisitos:
Implemente todas as rotas necessárias para realizar as operações CRUD.
Utilize um banco de dados SQLite para armazenar os dados dos usuários.
Garanta que a API seja segura, incluindo autenticação básica (ex.: token JWT).


### Resposta

Começando pela organização:

Organizei inicialmente a API em algumas camadas:

- `api` -> tudo relacionado ao FastAPI fica aqui. É a porta de entrada da aplicação.
- `domain` -> entidades e objetos do domínio.
- `infra` -> tudo relacionado a integrações externas, como banco de dados.
- `services` -> camada de negócio, onde ficam as aplicações de negócio das operações. Como é um CRUD, acabou ficando mais como uma camada de execução do repository.
- `auth` -> camada de autenticação do usuário.

Obs: várias decisões arquiteturais eu tomei apenas para deixar a API o mais simples possível, mas ainda minimamente organizada. Se fosse um projeto para operar com mais programadores, com certeza a organização subiria de nível, e muitas das decisões que tomei aqui não seriam feitas da mesma forma.

---

Começando pela camada de `domain`:

Iniciei criando os objetos do sistema. No caso, temos apenas o `User`. Como não temos mais entidades por enquanto, a camada de domínio ficou bem simples e direta.

A ideia aqui foi concentrar apenas aquilo que representa o domínio da aplicação, sem acoplar regra de banco, framework ou detalhes externos. Mesmo sendo um projeto pequeno, manter essa separação ajuda a deixar mais claro o que pertence ao sistema de fato e o que pertence à infraestrutura ao redor dele.

---

Na camada de `infra`:

Aqui já começa a integração com o mundo externo. Comecei criando a URL de conexão e um objeto simples que armazena essa URL do banco de dados.

Normalmente essa URL seria buscada do `.env`, mas como estamos tratando apenas com SQLite e a ideia era manter o projeto simples, decidi deixar mockado mesmo.

Depois disso, temos a engine do banco. Essa engine é o objeto que iremos usar para criar nossas sessões com o banco de dados. É bem simples: importamos o banco, criamos o objeto de engine e pronto.

Após isso, criei o `SessionManager`. Esse objeto é muito importante para todas as operações com banco de dados. Ele é quem vai dizer se um commit é válido ou não, além de funcionar como nosso objeto de integridade de dados.

Ele pode aceitar várias operações ao mesmo tempo, aceitar tudo ou rejeitar tudo. Isso é excelente principalmente quando usamos muitas transações que precisam ser processadas juntas e nenhuma delas pode falhar, porque se uma falhar no meio, dá muito ruim.

Esse objeto usa o padrão `Unit of Work`, que é ótimo para esse tipo de problema.

Com esses objetos criados, parti para o repository. O repository só precisa de uma injeção de dependência, que é o próprio `SessionManager`, usado para gerenciar nossas sessões e transações.

Ele é bem simples e direto, não tem muito o que comentar sobre ele. A responsabilidade dele ficou focada em acessar e manipular os dados, sem misturar regra de negócio ou detalhes da camada de API.

---

Na camada de `services`:

Após finalizar a camada de infra, parti para a camada de services.

Como não temos muitas operações de negócio, eu reduzi bastante essa camada. É apenas um CRUD simples. Então criamos nossos services e DTOs.

DTO é importante e eu sempre uso nas rotas de caso de uso, já que funcionam como contratos de entrada e saída. Para esse projeto em específico, usei os DTOs nas rotas para validar o corpo das requests, incluindo `body`, `path` e `query`.

Normalmente eu não colocaria dessa forma, mas como disse, queria algo direto. Então deixei aqui. Ainda assim, não considero essa a forma mais correta ou mais sábia em um cenário mais robusto.

Em uma API maior, provavelmente eu separaria melhor os DTOs por contexto, talvez distinguindo DTOs de entrada, saída, validação externa e objetos internos de aplicação. Mas para esse caso, manter tudo mais próximo deixou o fluxo mais simples de entender.

---

Na camada de `api`:

Essa camada é praticamente a porta de entrada do sistema. Aqui temos as rotas, o próprio servidor do FastAPI e o mapeamento de erros, que transforma erros internos em algo mapeado para a resposta HTTP.

A decisão de colocar esse mapeamento aqui não é a mais sábia arquiteturalmente, mas funciona bem levando em consideração a simplicidade do problema.

Também temos o adapter, uma peça importante do sistema. Ele transforma a request do FastAPI em uma request HTTP interna do sistema.

Como ele é genérico, ele valida os formatos dentro das rotas. É bem interessante e é uma prática recente que venho adotando nas minhas APIs comerciais.

Temos também os controllers, que realizam as operações usando os casos de uso. Eles coletam a `HttpRequest` do sistema e devolvem uma `HttpResponse`.

O `config` contém o singleton da `ENGINE`, que será utilizado para gerenciar sessões juntamente com o `SessionManager`.

Por fim, temos os composers, que são factories responsáveis por criar nossos controllers.

Também temos as rotas, que usam praticamente tudo da camada de API e também parte dos services, principalmente os DTOs, usados para exigir e documentar os campos das rotas.

---

Na camada de `auth`:

A camada de autenticação concentra toda a lógica de identificação e autorização do usuário — geração e validação de tokens JWT, hash e verificação de senhas com bcrypt, e a chave secreta usada para assinar os tokens.

A ideia é evitar misturar regras de autenticação diretamente dentro das rotas ou dos services principais. Mesmo sendo algo simples (validação de credenciais contra variáveis de ambiente e proteção das rotas via dependência `protect`), essa separação mantém o fluxo organizado e facilita evoluir a autenticação depois — por exemplo, migrar para múltiplos usuários, OAuth ou refresh tokens.

Aqui entram responsabilidades como validação de credenciais, geração e decodificação de tokens, o middleware de proteção das rotas e qualquer regra diretamente relacionada à identificação do usuário.

(Aqui foi feito simples, somente credenciais de admin usando o .env, sem precisar criar usuários)

---

De forma geral:

A estrutura ficou simples, mas com uma separação mínima de responsabilidades.

O `domain` representa os objetos principais do sistema.

A `infra` cuida da comunicação com o mundo externo, principalmente banco de dados.

A `services` concentra os fluxos de aplicação e regras mais próximas do negócio.

A `api` expõe o sistema para fora usando FastAPI, controllers, adapters, rotas e composers.

E a `auth` isola a parte de autenticação para não espalhar essa responsabilidade pelo resto do projeto.

Como o problema é pequeno e basicamente um CRUD, algumas decisões foram feitas pensando mais em clareza e velocidade do que em arquitetura ideal. Em um cenário maior, com mais regras, mais programadores e mais manutenção, eu certamente separaria melhor algumas responsabilidades e evitaria alguns atalhos que aqui foram intencionais.

## Parte 3: Integração de Modelos de IA

Implemente uma rota adicional na API criada na Parte 1 que permita a predição de um modelo de IA simples. Use um modelo treinado para prever valores numéricos a partir de uma entrada de dados.

Requisitos:
- Salve o modelo treinado utilizando a biblioteca joblib ou pickle.
- Crie uma rota na API que receba dados de entrada, utilize o modelo para fazer a predição e retorne o resultado.


### Resposta

vamos criar o modelo primeiro:

In [6]:
from sklearn.ensemble import RandomForestClassifier # O modelo de forest, um modelo bem simples e robusto
from sklearn.model_selection import train_test_split # Separar test e traino
from sklearn.preprocessing import MinMaxScaler # Para padronizar dados
from sklearn.metrics import accuracy_score # Para avaliar o modelo
import numpy as np
from random import randint
import joblib

sample_space = 50000
X = np.array([np.array([randint(-5000, 5000), randint(-5000, 5000)]) for _ in range(sample_space)])
y = np.array([1 if (X[i][0] + X[i][1]) > 0 else 0 for i in range(sample_space)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

joblib.dump(model, '../models/random_forest_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')

Accuracy: 1.00


['../models/scaler.joblib']

Criei um modelo simples só para dizer se a soma de um número é positivo ou negativo

Pasta `ml` dentro de `src`:

Essa pasta centraliza o carregamento dos artefatos de Machine Learning utilizados pela aplicação.

Ela importa e instancia tanto o `scaler` quanto o próprio `modelo`, armazenando-os em variáveis compartilhadas no padrão singleton. Isso evita que o mesmo modelo seja carregado várias vezes em memória a cada requisição ou importação, reduzindo consumo de recursos e melhorando a eficiência da aplicação.

Atualmente, o modelo utilizado é simples e serve apenas como suporte para os testes das rotas, permitindo validar o fluxo de predição sem depender ainda de um modelo final ou mais complexo.


Além disso, foram criados os objetos necessários para a estrutura das rotas padrão da aplicação, incluindo service, controller e composer/factory.

Essa separação foi necessária porque a aplicação segue esse padrão arquitetural para permitir que o adapter consiga intermediar corretamente a comunicação entre a rota HTTP e a camada de controle. Dessa forma, cada responsabilidade fica isolada: o service concentra a lógica de negócio, o controller organiza a entrada e saída da requisição, e o composer/factory instancia e conecta as dependências necessárias para o funcionamento da rota.

Como os objetos do `scikit-learn` não são assíncronos, eles podem bloquear o event loop do FastAPI durante a execução da predição. Para evitar esse bloqueio, foi criada uma `ThreadPool` utilizando o módulo `concurrent.futures` do Python.

Com isso, as chamadas de predição são executadas em threads separadas, permitindo que a aplicação continue respondendo outras requisições enquanto o modelo processa os dados. Esse ajuste resolve um gargalo de performance nesse caso específico e permite executar previsões concorrentes, como 4 ou mais predições ao mesmo tempo, dependendo da configuração da pool e dos recursos disponíveis.

Para uma aplicação real em produção, porém, essa solução isolada provavelmente não seria suficiente. O ideal seria separar o serviço de Machine Learning em uma aplicação ou servidor próprio e trabalhar com filas de processamento. Dessa forma, seria possível escalar horizontalmente: caso fosse necessário aumentar a capacidade de processamento, bastaria subir novas instâncias/servidores do serviço responsável pelas predições.

## Parte 4: Docker e Orquestração de Contêineres

Crie um arquivo Dockerfile para a aplicação desenvolvida nas Partes 2 e 3. Adicionalmente, crie um arquivo docker-compose.yml para orquestrar a aplicação e o banco de dados.

Requisitos:
- O Dockerfile deve configurar o ambiente necessário para rodar a aplicação.
- O arquivo docker-compose.yml deve definir serviços para a aplicação e o banco de dados.
- Inclua instruções de como construir e rodar os contêineres.


### Resposta